# Serve a model with FastAPI

**JobReady AI lab.** Run each cell with Shift+Enter. Try the exercises before opening the solutions at the end.

**You will:**
- Train and save a scikit-learn pipeline
- Wrap it in a validated FastAPI endpoint
- Test it and simulate drift monitoring

In [ ]:
!pip -q install fastapi httpx
import numpy as np, pandas as pd, joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(0)
n = 3000
df = pd.DataFrame({'tenure': rng.integers(1, 72, n), 'monthly_charges': rng.normal(70, 25, n).clip(15, 150), 'contract': rng.choice(['month', 'year', 'two_year'], n, p=[.55, .3, .15])})
churn = (rng.random(n) < 1 / (1 + np.exp(-(-1 + 0.02 * df.monthly_charges - 0.05 * df.tenure + (df.contract == 'month') * 1.2)))).astype(int)

## 1. A pipeline: preprocessing + model in one object

In [ ]:
pre = ColumnTransformer([('num', StandardScaler(), ['tenure', 'monthly_charges']), ('cat', OneHotEncoder(handle_unknown='ignore'), ['contract'])])
pipe = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000))]).fit(df, churn)
joblib.dump(pipe, 'pipeline.joblib')

## 2. The API

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import Literal
app = FastAPI()
model = joblib.load('pipeline.joblib')
class Customer(BaseModel):
    tenure: int = Field(ge=0, le=120)
    monthly_charges: float = Field(gt=0, lt=1000)
    contract: Literal['month', 'year', 'two_year']
@app.post('/predict')
def predict(c: Customer):
    p = model.predict_proba(pd.DataFrame([c.model_dump()]))[0, 1]
    return {'churn_probability': round(float(p), 4), 'model_version': '2026-10-01'}

## 3. Test it

In [ ]:
from fastapi.testclient import TestClient
client = TestClient(app)
print(client.post('/predict', json={'tenure': 3, 'monthly_charges': 99, 'contract': 'month'}).json())
print(client.post('/predict', json={'tenure': -5, 'monthly_charges': 99, 'contract': 'month'}).status_code)   # 422: rejected input

## 4. Drift check with the Population Stability Index (PSI)

In [ ]:
def psi(expected, actual, bins=10):
    cuts = np.quantile(expected, np.linspace(0, 1, bins + 1)); cuts[0], cuts[-1] = -np.inf, np.inf
    e = np.histogram(expected, cuts)[0] / len(expected); a = np.histogram(actual, cuts)[0] / len(actual)
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
    return float(((a - e) * np.log(a / e)).sum())
live_same = rng.normal(70, 25, 2000)
live_shift = rng.normal(85, 25, 2000)
print('PSI same', round(psi(df.monthly_charges, live_same), 3), ' PSI shifted', round(psi(df.monthly_charges, live_shift), 3), '(>0.2 = investigate)')

## Exercises
1. Add a `/health` endpoint that returns the model version.
2. Write a Dockerfile for this app (uvicorn on port 8080).

In [ ]:
# your work here

## Solutions
```python
@app.get('/health')
def health(): return {'status': 'ok', 'model_version': '2026-10-01'}
```
```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080"]
```